# Extract points

Utility notebook for generating study_area_polygons, unlabelled points, and prospectivity grid points. Created to test implementation of mantle extraction without necessitating the full grid extraction pipeline to work.

## Notebook options

These cells set some of the important variables and definitions used throughout the notebook.

In [1]:
config_file = "config/.run_config.yml"

In [2]:
from lib.load_params import get_params
from pathlib import Path

params = get_params(config_file, notebook="00bb")

plate_model_name = params["plate_model"]["plate_model_name"]
use_provided_plate_model = params["plate_model"]["use_provided_plate_model"]

# =====================
# Filestructure
# =====================

# Parent data directory
parent_data_dir = Path(params["data_dir"])

# Data directory for chosen reconstruction
recon_data_dir = parent_data_dir / plate_model_name

# Directory for raster inputs
data_dir = recon_data_dir / "rasters"

# Extracted data directory
output_dir = (
    recon_data_dir /
    "extracted_data" /
    f"{params["reference_feature"]}_{params["study_zone_buffer"]:.1f}_deg_buffer"
)

# Plate model directory
plate_model_dir = recon_data_dir / "plate_model"

# CSV file with known deposits; columns:
# lon, lat, age (Ma), label, source
deposits_filename = parent_data_dir / "deposits" / params["deposits_filename"]

# If desired, categorise deposits according to location
# Should be a shapefile or GeoJSON containing polygons
# with a 'region' attribute
regions_filename = parent_data_dir / params["regions_filename"]

# Initialise filestructure
parent_data_dir.mkdir(parents=True, exist_ok=True)
recon_data_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

# =====================
# Run parameters
# =====================

# Number of processes to use
n_jobs = params["n_jobs"]

# Overwrite any existing output files
overwrite = params["overwrite_output"]

# Control verbosity level of logging output
verbose = params["verbose"]

# Timespan for analysis
min_time = params["timespan"]["min"]
max_time = params["timespan"]["max"]
times = range(min_time, max_time + 1)

# Number of unlabelled points to generate
num_unlabelled = params["num_unlabelled"]  # per timestep

# Resolution of output grids
grid_resolution = params["grid_resolution"]

# Random seed for reproducibility
random_seed = params["random_seed"]


## Notebook setup

Imports, definitions, etc.

### Imports

In [3]:
import os
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    from gplately.tools import plate_isotherm_depth

from lib.assign_regions import assign_regions
from lib.calculate_convergence import run_calculate_convergence
from lib.check_files import (
    check_source_data,
    check_plate_model,
)
from lib.combine_point_data import combine_point_data
from lib.coregister_combined_point_data import run_coregister_combined_point_data
from lib.coregister_crustal_thickness import run_coregister_crustal_thickness
from lib.coregister_ocean_rasters import (
    extract_subducted_thickness,
    run_coregister_ocean_rasters,
)
from lib.create_study_area_polygons import run_create_study_area_polygons
from lib.erodep import calculate_erodep
from lib.generate_unlabelled_points import generate_unlabelled_points
from lib.misc import calculate_slab_flux, calculate_carbon
from lib.plate_models import get_plate_reconstruction
from lib.pu import generate_grid_points
from lib.slab_dip import calculate_slab_dip
from lib.water import calculate_water_thickness

from lib.create_study_area_polygons import run_create_study_area_polygons
from lib.erodep import calculate_erodep
from lib.misc import calculate_slab_flux, calculate_carbon
from lib.plate_models import get_plate_reconstruction
from lib.slab_dip import calculate_slab_dip
from lib.water import calculate_water_thickness

# Suppress occasional joblib warnings
%env PYTHONWARNINGS=ignore::UserWarning
warnings.simplefilter("ignore", UserWarning)

env: PYTHONWARNINGS=ignore::UserWarning


### Input and output files

If necessary, the plate model will be downloaded:

In [4]:
if use_provided_plate_model:
    check_plate_model(plate_model_dir, verbose=True)
    plate_model_name = None
plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

2026-03-22,17:16:46 - pmm - WARNING - Unable to fetch https://repo.gplates.org/webdav/pmm/config/models_v2.json.
2026-03-22,17:16:46 - pmm - WARNING - Unable to fetch https://www.earthbyte.org/webdav/pmm/config/models_v2_eb.json.
2026-03-22,17:16:46 - pmm - WARNING - Unable to fetch https://portal.gplates.org/static/pmm/config/models_v2_gp.json.


*^^ Runtime: 1m 39s*

The following input directories are all relative to `data_dir`:

In [5]:
# Handle relative file/directory paths

study_area_dir = output_dir / "study_area_polygons"
unlabelled_points_filename = output_dir / "unlabelled_points.csv"
combined_points_filename = output_dir / "combined_points.csv"
training_points_filename = output_dir / "training_data_global.csv"
grid_points_filename = output_dir / "grid_points.csv"
grid_data_filename = output_dir / "grid_data.csv"

### Create study area polygons along subduction zones

Here we define our study area as all points on the overriding plate within a certain distance of the subduction zone (by default, $6 \degree, \approx 660\mathrm{km}$)

In [14]:
buffer_distance = params["study_zone_buffer"] # 6.0

if overwrite or not os.path.isdir(study_area_dir):
    run_create_study_area_polygons(
        nprocs=n_jobs,
        times=times,
        plate_reconstruction=plate_model,
        output_dir=study_area_dir,
        buffer_distance=buffer_distance,
        verbose=verbose,
        return_output=False,
    )

*^^^ Runtime ~14 mins*

### Generate random unlabelled data points

The unlabelled set is created by generating uniformly-distributed random points within the polygons created in the previous cell. To change the number of points generated at each timestep, modify the `num_unlabelled` parameter defined earlier.

In [ ]:
if overwrite or not os.path.isfile(unlabelled_points_filename):
    unlabelled = generate_unlabelled_points(
        times=times,
        input_dir=study_area_dir,
        num=num_unlabelled,
        threads=n_jobs,
        seed=random_seed,
        plate_reconstruction=plate_model,
        verbose=verbose,
        output_file=unlabelled_points_filename
    )
else:
    unlabelled = pd.read_csv(unlabelled_points_filename)

*^^^ Runtime ~14 mins*

### Combine labelled deposit/non-deposit data with random unlabelled data

The function below wrangles the points generated in the previous cell into the same format as the deposit location data.

In [ ]:
if overwrite or not os.path.isfile(combined_points_filename):
    combined_points = combine_point_data(
        deposit_data=deposits_filename,
        unlabelled_data=unlabelled,
        plate_reconstruction=plate_model,
        study_area_dir=study_area_dir,
        min_time=min(times),
        max_time=max(times),
        n_jobs=n_jobs,
        verbose=verbose,
        output_filename=combined_points_filename
    )
    del unlabelled
    combined_points = combined_points.dropna(subset=["present_lon", "present_lat"])
else:
    combined_points = pd.read_csv(combined_points_filename)

Preparing labelled data...
Loading deposit data from: /scratch/xd2/me5758/PUB-framework-Alfonso/data/deposits/deposits-Etherington.csv
[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:  1.0min
[Parallel(n_jobs=8)]: Done 488 tasks      | elapsed:  1.1min
[Parallel(n_jobs=8)]: Done 674 out of 689 | elapsed:  1.1min remaining:    1.5s
[Parallel(n_jobs=8)]: Done 689 out of 689 | elapsed:  1.1min finished
[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 out of   8 | elapsed:   54.9s remaining:  2.7min
[Parallel(n_jobs=8)]: Done   8 out of   8 | elapsed:  1.6min finished
Done.
Preparing unlabelled data...
[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 out of   8 | elapsed:   55.7s remaining:  2.8min
[Parallel(n_jobs=8)]: Done   8 out of   8 | elapsed:  1.5min finished
Done.


### Assign training point data to regions

To divide the data into individual regions for the later analysis, we use the `regions_filename` defined earlier, if desired.

In [11]:
if regions_filename is not None and os.path.isfile(regions_filename):
    points = gpd.GeoSeries.from_xy(
        combined_points["present_lon"],
        combined_points["present_lat"],
        index=combined_points.index,
    )
    combined_points["region"] = assign_regions(
        points,
        regions=regions_filename,
    )
    del points

### Save to file

Finally, we write the dataset to a CSV file.

In [12]:
combined_points.to_csv(training_points_filename, index=False)
if combined_points_filename is not None:
    combined_points.to_csv(combined_points_filename, index=False)

print(combined_points.groupby(["region", "label"]).size())
del combined_points

region          label     
East Asia       negative        14
                positive       159
                unlabelled    7729
North America   negative        61
                positive       293
                unlabelled    8104
Other           negative       214
                positive         2
                unlabelled    6476
South America   negative      1389
                positive       275
                unlabelled    7845
Southeast Asia  negative         4
                positive       145
                unlabelled    5457
Tethys          negative        21
                positive       481
                unlabelled    6507
dtype: int64


### Generate grid points

The following function generates the grid of points at `grid_resolution`-degree resolution.

In [ ]:
if overwrite or not os.path.isfile(grid_points_filename):
    grid_points = generate_grid_points(
        times=times,
        resolution=grid_resolution,
        polygons_dir=study_area_dir,
        plate_reconstruction=plate_model,
        n_jobs=n_jobs,
        verbose=verbose,
    )
    grid_points = grid_points.dropna(subset=["present_lon", "present_lat"])
    grid_points.to_csv(grid_points_filename, index=False)
else:
    grid_points = pd.read_csv(grid_points_filename)

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


[Parallel(n_jobs=8)]: Done   2 out of   8 | elapsed: 141.7min remaining: 425.1min


KeyboardInterrupt: 

*^^^ Runtime: `[Parallel(n_jobs=8)]: Done   2 out of   8 | elapsed: 141.7min remaining: 425.1min`*

### Assign grid data to regions

To divide the data into individual regions for the later analysis, we use the `regions_filename` defined earlier, if desired.

In [ ]:
if regions_filename is not None and os.path.isfile(regions_filename):
    points = gpd.GeoSeries.from_xy(
        grid_points["present_lon"],
        grid_points["present_lat"],
        index=grid_points.index,
    )
    grid_points["region"] = assign_regions(
        points,
        regions=regions_filename,
    )
    del points

### Save to file

Finally, we write the grid (point) data to a CSV file.

In [ ]:
grid_points.to_csv(grid_points_filename, index=False)
if grid_points_filename is not None:
    grid_points.to_csv(grid_points_filename, index=False)